## Vectorisation des paragraphes 

Format avec modeles Em de base et fine tuné 

In [1]:
!pip uninstall torch torchvision torchaudio -y
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124

Found existing installation: torch 2.6.0+cu124
Uninstalling torch-2.6.0+cu124:
  Successfully uninstalled torch-2.6.0+cu124
Found existing installation: torchvision 0.21.0+cu124
Uninstalling torchvision-0.21.0+cu124:
  Successfully uninstalled torchvision-0.21.0+cu124
Found existing installation: torchaudio 2.6.0+cu124
Uninstalling torchaudio-2.6.0+cu124:
  Successfully uninstalled torchaudio-2.6.0+cu124
Looking in indexes: https://download.pytorch.org/whl/cu124
  Using cached torch-2.6.0%2Bcu124-cp311-cp311-win_amd64.whl.metadata (28 kB)
  Using cached torchvision-0.21.0%2Bcu124-cp311-cp311-win_amd64.whl.metadata (6.3 kB)
  Using cached torchaudio-2.6.0%2Bcu124-cp311-cp311-win_amd64.whl.metadata (6.8 kB)
Using cached torch-2.6.0%2Bcu124-cp311-cp311-win_amd64.whl (2532.4 MB)
Using cached torchvision-0.21.0%2Bcu124-cp311-cp311-win_amd64.whl (6.1 MB)
Using cached torchaudio-2.6.0%2Bcu124-cp311-cp311-win_amd64.whl (4.2 MB)

   ---------------------------------------- 0/3 [torch]
   ------

In [2]:
import pandas as pd
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

c:\Users\natal\OneDrive\Bureau\COURS_TELECOM\Amiad Hackathon\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# ── Chemins ───────────────────────────────────────────────────────────────────
CSV_PATH        = r"principale\data-challenge-amiad-telecom-monamocost-frontend\Experimental\Cluster_run_test\DATA\df_paragraphe_final.csv"          # chemin vers ton fichier CSV
FINETUNED_PATH  = "YYY"          # chemin vers les poids du modèle fine-tuné
OUTPUT_BASE     = "embeddings_base.npz"       # sortie vectorisation modèle de base
OUTPUT_FINETUNED= "embeddings_finetuned.npz"  # sortie vectorisation modèle fine-tuné

# ── Paramètres ────────────────────────────────────────────────────────────────
MODEL_NAME  = "intfloat/multilingual-e5-large"
BATCH_SIZE  = 32
PREFIX      = "passage: "   # préfixe attendu par e5 pour les passages à indexer
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Device : {DEVICE}")

Device : cuda


In [4]:
def load_corpus(csv_path: str) -> pd.DataFrame:
    """Charge le CSV et affiche un résumé."""
    df = pd.read_csv(csv_path)
    assert {"paragraphe", "nom_du_fichier", "page"}.issubset(df.columns), \
        "Colonnes attendues : paragraphe, nom_du_fichier, page"
    df = df.dropna(subset=["paragraphe"]).reset_index(drop=True)
    print(f"Corpus chargé : {len(df)} paragraphes | {df['nom_du_fichier'].nunique()} fichiers")
    return df


def add_prefix(texts: list[str], prefix: str) -> list[str]:
    """Ajoute le préfixe e5 à chaque texte."""
    return [prefix + t for t in texts]


def vectorize(
    model: SentenceTransformer,
    texts: list[str],
    batch_size: int = 32,
    prefix: str = "passage: ",
    device: str = "cpu",
) -> np.ndarray:
    """
    Vectorise une liste de textes par batchs.
    Retourne un array numpy (n_textes, dim_embedding), normalisé L2.
    """
    model.to(device)
    model.eval()

    prefixed = add_prefix(texts, prefix)
    all_embs = []

    with torch.no_grad():
        for i in tqdm(range(0, len(prefixed), batch_size), desc="Vectorisation"):
            batch = prefixed[i : i + batch_size]
            embs  = model.encode(
                batch, 
                batch_size=batch_size,
                normalize_embeddings=True,
                convert_to_numpy=True,
                device=device,
                show_progress_bar=False,
            )
            all_embs.append(embs)

    return np.vstack(all_embs)


def save_embeddings(embeddings: np.ndarray, df: pd.DataFrame, output_path: str):
    """Sauvegarde embeddings + métadonnées dans un fichier .npz"""
    np.savez(
        output_path,
        embeddings     = embeddings,
        paragraphes    = df["paragraphe"].values,
        noms_fichiers  = df["nom_du_fichier"].values,
        pages          = df["page"].values,
    )
    print(f"Sauvegardé → {output_path}  |  shape : {embeddings.shape}")

#### Modele de base

In [5]:
df = load_corpus(CSV_PATH)

model_base = SentenceTransformer(MODEL_NAME)

embeddings_base = vectorize(
    model     = model_base,
    texts     = df["paragraphe"].tolist(),
    batch_size= BATCH_SIZE,
    prefix    = PREFIX,
    device    = DEVICE,
)

save_embeddings(embeddings_base, df, OUTPUT_BASE)

FileNotFoundError: [Errno 2] No such file or directory: 'principale\\data-challenge-amiad-telecom-monamocost-frontend\\Experimental\\Cluster_run_test\\DATA\\df_paragraphe_final.csv'

#### Modele finetuné

In [ ]:
model_finetuned = SentenceTransformer(FINETUNED_PATH)

embeddings_finetuned = vectorize(
    model     = model_finetuned,
    texts     = df["paragraphe"].tolist(),
    batch_size= BATCH_SIZE,
    prefix    = PREFIX,
    device    = DEVICE,
)

save_embeddings(embeddings_finetuned, df, OUTPUT_FINETUNED)